# 03 — Baseline: Elo Rating
CS2 Match Outcome Predictor & Seeding Engine

Computes a time-ordered Elo rating for every team, using only
(date, team1_name, team2_name, winner) — no other features.

This serves as:
1. The project baseline (compared against Logistic Regression / XGBoost later)
2. A feature (elo_diff) fed into the main model in 04_model_training.ipynb
3. The final per-team rating used for seeding in 06_demo.ipynb

In [ ]:
import os

def find_repo_root(marker='requirements.txt'):
    path = os.getcwd()
    while True:
        if os.path.exists(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find repo root (looking for '{marker}')")
        path = parent

repo_root = find_repo_root()
os.chdir(repo_root)

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("Working directory set to:", os.getcwd())

In [1]:
import logging
import time

logging.Formatter.converter = lambda *args: time.localtime(time.time() + 5*3600)

logging.basicConfig(
    filename='data/03_baseline_elo.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    filemode='w',
    force=True
)

In [2]:
import pandas as pd

df = pd.read_csv('data/cleaned_matches.csv')
df['date'] = pd.to_datetime(df['date'])

print("Shape:", df.shape)
logging.info(f"Loaded cleaned_matches.csv, shape: {df.shape}")
df.head(3)

Shape: (6989, 55)


,match_id,hltv_match_id,date,tournament,winner,season,score_team1,score_team2,winner_map,loser_map,...,team1_totalwinrate,team2_totalwinrate,team1_totallossrate,team2_totallossrate,team1_online_winrate,team2_online_winrate,team1_lan_winrate,team2_lan_winrate,team1_overall_winrate,team2_overall_winrate
0,hltv_match_2371997,2371997,2024-05-15 06:00:00+00:00,BetBoom Dacha Belgrade 2024,team1,8,2,0,Ancient,Dust2,...,0.805556,0.400000,0.194444,0.600000,0.000000,0.442308,0.460317,0.379310,0.5,0.5
1,hltv_match_2372188,2372188,2024-05-15 08:00:00+00:00,RES Regional Series 4 Europe,team2,8,1,2,Nuke,Anubis,...,0.586207,0.431818,0.413793,0.568182,0.410714,0.398058,0.478261,0.285714,0.5,0.5
2,hltv_match_2372226,2372226,2024-05-15 08:30:00+00:00,CCT Season 2 Europe Series 4 Closed Qualifier,team2,8,1,2,Ancient,Vertigo,...,0.250000,0.428571,0.750000,0.571429,0.500000,0.506173,0.500000,0.000000,0.5,0.5


## 1. Elo update function

Standard Elo formula:
- Expected score: E_A = 1 / (1 + 10^((Elo_B - Elo_A) / 400))
- New rating: Elo_A_new = Elo_A + K * (S_A - E_A)

In [3]:
INITIAL_ELO = 1500
K = 32

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(elo_a, elo_b, score_a, k=K):
    exp_a = expected_score(elo_a, elo_b)
    return elo_a + k * (score_a - exp_a)

## 2. Compute Elo chronologically for every match

For each match (in date order):
- Look up both teams' current Elo (default 1500 if new)
- Record the PRE-MATCH Elo for both teams (used later as model features —
  this is what was "known" before the match happened, avoiding leakage)
- Update both teams' Elo based on the result

In [4]:
from collections import defaultdict

elo_ratings = defaultdict(lambda: INITIAL_ELO)

team1_elo_pre = []
team2_elo_pre = []

for idx, row in df.iterrows():
    t1, t2 = row['team1_name'], row['team2_name']
    elo1, elo2 = elo_ratings[t1], elo_ratings[t2]

    # store pre-match Elo as features
    team1_elo_pre.append(elo1)
    team2_elo_pre.append(elo2)

    # update based on result
    score1 = 1 if row['winner'] == 'team1' else 0
    new_elo1 = update_elo(elo1, elo2, score1)
    new_elo2 = update_elo(elo2, elo1, 1 - score1)

    elo_ratings[t1] = new_elo1
    elo_ratings[t2] = new_elo2

df['team1_elo_pre'] = team1_elo_pre
df['team2_elo_pre'] = team2_elo_pre
df['elo_diff'] = df['team1_elo_pre'] - df['team2_elo_pre']

logging.info(f"Computed Elo for {len(elo_ratings)} unique teams over {len(df)} matches")
print("Done. Sample:")
df[['date', 'team1_name', 'team2_name', 'winner', 'team1_elo_pre', 'team2_elo_pre', 'elo_diff']].head(20)

Done. Sample:


,date,team1_name,team2_name,winner,team1_elo_pre,team2_elo_pre,elo_diff
0,2024-05-15 06:00:00+00:00,Spirit,Aurora,team1,1500.000000,1500.000000,0.000000
1,2024-05-15 08:00:00+00:00,3DMAX,Zero Tenacity,team2,1500.000000,1500.000000,0.000000
2,2024-05-15 08:30:00+00:00,MASONIC,CPH Wolves,team2,1500.000000,1500.000000,0.000000
3,2024-05-15 08:45:00+00:00,HEROIC,MIBR,team1,1500.000000,1500.000000,0.000000
4,2024-05-15 09:30:00+00:00,EYEBALLERS,9INE,team1,1500.000000,1500.000000,0.000000
5,2024-05-15 12:50:00+00:00,BetBoom,Falcons,team2,1500.000000,1500.000000,0.000000
6,2024-05-15 13:00:00+00:00,Imperial,Fluxo,team1,1500.000000,1500.000000,0.000000
7,2024-05-15 14:00:00+00:00,B8,PARIVISION,team2,1500.000000,1500.000000,0.000000
8,2024-05-15 14:00:00+00:00,GUN5,777,team1,1500.000000,1500.000000,0.000000
9,2024-05-15 16:00:00+00:00,9z,Sharks,team1,1500.000000,1500.000000,0.000000


## 3. Sanity checks

- Elo should correlate with winning (higher elo_diff → team1 more likely to win)
- Final Elo distribution should look reasonable (not exploding/collapsing)

In [5]:
df['team1_won'] = (df['winner'] == 'team1').astype(int)

# Simple check: average elo_diff when team1 won vs lost
print("Mean elo_diff when team1 WON:", df.loc[df['team1_won']==1, 'elo_diff'].mean())
print("Mean elo_diff when team1 LOST:", df.loc[df['team1_won']==0, 'elo_diff'].mean())

logging.info(f"Mean elo_diff (team1 won): {df.loc[df['team1_won']==1, 'elo_diff'].mean():.2f}")
logging.info(f"Mean elo_diff (team1 lost): {df.loc[df['team1_won']==0, 'elo_diff'].mean():.2f}")

Mean elo_diff when team1 WON: 32.36847122289483
Mean elo_diff when team1 LOST: -8.967985219806033


In [6]:
final_elo = pd.Series(elo_ratings).sort_values(ascending=False)
print("Top 10 teams by final Elo:")
print(final_elo.head(10))
print("\nElo stats:")
print(final_elo.describe())

logging.info(f"Final Elo range: {final_elo.min():.1f} - {final_elo.max():.1f}")

Top 10 teams by final Elo:
Vitality         1948.662291
Spirit           1861.844625
MOUZ             1824.193444
Falcons          1819.399184
FURIA            1814.108727
The MongolZ      1793.862739
PARIVISION       1770.871181
Natus Vincere    1767.450459
ESC              1756.797656
fnatic           1737.512473
dtype: float64

Elo stats:
count     331.000000
mean     1500.000000
std        94.427867
min      1285.625752
25%      1446.419361
50%      1481.181754
75%      1533.056515
max      1948.662291
dtype: float64


## 4. Save outputs

- `matches_with_elo.csv`: full match dataset + Elo features (for 04_model_training.ipynb)
- `elo_ratings.csv`: final per-team Elo, used later for seeding (06_demo.ipynb)

In [7]:
df.to_csv('data/matches_with_elo.csv', index=False)
logging.info(f"Saved matches_with_elo.csv, shape: {df.shape}")

final_elo_df = pd.DataFrame({
    'team_name': final_elo.index,
    'elo_rating': final_elo.values
})
final_elo_df.to_csv('models/elo_ratings.csv', index=False)
logging.info(f"Saved elo_ratings.csv, {len(final_elo_df)} teams")

print("Saved both files.")

Saved both files.


In [8]:
try:
    from google.colab import files
    files.download('data/matches_with_elo.csv')
    files.download('models/elo_ratings.csv')
    files.download('data/03_baseline_elo.log')
except ImportError:
    print("Not running in Colab — files saved locally.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>